In [1]:
import pandas as pd
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from datasets import Dataset
from transformers import BertTokenizer, TrainingArguments, Trainer

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")

In [5]:
print(df.columns)
print(df.shape)

Index(['verse_id', 'song_id', 'ori_track_name', 'clean_track_name',
       'all_artists', 'primary_artist', 'artist_genres', 'main_genre',
       'explicit', 'section', 'verse', 'language', 'language.1', 'confidence',
       'confidence.1', 'label'],
      dtype='str')
(1000, 16)


In [6]:
# Class balance check
print(df['label'].value_counts())

label
0    1000
Name: count, dtype: int64


In [7]:
# Select only the columns we need
df = df[['verse', 'label']]

In [8]:
# Convert to Hugging Face format
dataset = Dataset.from_pandas(df)

In [9]:
# Fixes the [WinError 3] by explicitly setting a local cache directory
cache_dir = "C:/Users/User/Documents/devanasokan_fyp/huggingface_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

In [10]:
# Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir

# This turns off the annoying symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [11]:
# Initiate with the cache path
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', cache_dir=cache_dir)

In [12]:
def preprocess_function(examples):
    # This creates the 'input_ids' and 'attention_mask' BERT needs
    return tokenizer(examples["verse"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 1679.10 examples/s]


In [13]:
# Split the dataset into training and testing sets
# Different ratios are tested
full_dataset = tokenized_dataset.train_test_split(test_size=0.3, seed=42)

In [14]:
print(full_dataset) 
# If it shows {'train': ..., 'test': ...}, it is already split!

DatasetDict({
    train: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 700
    })
    test: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 300
    })
})


In [15]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased" # Or any model from the Hugging Face Hub

# 1. Load the tokenizer (must match the model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load the model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4213.35it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
import evaluate
metric = evaluate.load("accuracy")

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # argmax picks the highest probability (0 or 1)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [18]:
# Set training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Folder where checkpoints are saved
    eval_strategy="epoch",     # Run evaluation after every epoch
    save_strategy="epoch",           # Save model after every epoch
    learning_rate=2e-5,              # Standard BERT fine-tuning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    num_train_epochs=3,              # Total passes through the data
    weight_decay=0.01,               # Regularization to prevent overfitting
    load_best_model_at_end=True,     # Keeps the best version of the model
    fp16=torch.cuda.is_available()  # Use Mixed Precision if on GPU for 2x speed
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

print(trainer.compute_metrics)

<function compute_metrics at 0x00000181D82168E0>


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.004726,1.000000
2,No log,0.002137,1.000000
3,No log,0.001770,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=132, training_loss=0.04000371152704412, metrics={'train_runtime': 62.0788, 'train_samples_per_second': 33.828, 'train_steps_per_second': 2.126, 'total_flos': 278181537177600.0, 'train_loss': 0.04000371152704412, 'epoch': 3.0})

In [21]:
history = pd.DataFrame(trainer.state.log_history)
print(history)

   eval_loss  eval_accuracy  eval_runtime  eval_samples_per_second  \
0   0.004726            1.0        2.4793                  121.001   
1   0.002137            1.0        2.1982                  136.477   
2   0.001770            1.0        2.5038                  119.816   
3        NaN            NaN           NaN                      NaN   

   eval_steps_per_second  epoch  step  train_runtime  \
0                  7.663    1.0    44            NaN   
1                  8.644    2.0    88            NaN   
2                  7.588    3.0   132            NaN   
3                    NaN    3.0   132        62.0788   

   train_samples_per_second  train_steps_per_second    total_flos  train_loss  
0                       NaN                     NaN           NaN         NaN  
1                       NaN                     NaN           NaN         NaN  
2                       NaN                     NaN           NaN         NaN  
3                    33.828                   2.

In [26]:
# Confusion matrix and classification report
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
classification_report_output = classification_report(full_dataset["test"]["label"], np.argmax(trainer.predict(full_dataset["test"]).predictions, axis=-1))
print(classification_report_output)

Classification Report:


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       300

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [23]:
# Save the version currently in the trainer's brain
trainer.save_model("./my_final_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]


In [24]:
# TEST MODEL

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load the model and tokenizer from your local folder
path = "./my_final_model"
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Ensure you saved the tokenizer there too!

# Create a 'pipeline' (the easiest way to use the model)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6103.54it/s]


In [25]:
# Test it on a new sentence
result = classifier("kiss it from my lips")
print(result)

[{'label': 'LABEL_1', 'score': 0.7351884841918945}]


In [34]:
import accelerate
print(accelerate.__version__)

1.13.0


In [35]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.13.0
Transformers version: 5.3.0


In [36]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
